# 02 - Verify the public LiteLLM proxy

This notebook loads the generated connection details and checks the public health endpoint and gateway model discovery endpoint.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../env/.env')
required = ['LITELLM_PROXY_URL', 'LITELLM_MASTER_KEY', 'LITELLM_MODEL_ALIAS']
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing {missing}; run 01_deploy_infra.ipynb first.')
proxy_url = os.environ['LITELLM_PROXY_URL'].rstrip('/')
proxy_key = os.environ['LITELLM_MASTER_KEY']
model_alias = os.environ['LITELLM_MODEL_ALIAS']
print(f'Proxy URL: {proxy_url}')
print(f'Model alias: {model_alias}')

## Step 1 - Check health and model discovery

In [ ]:
import requests

health = requests.get(f'{proxy_url}/health/liveliness', timeout=30)
health.raise_for_status()
models = requests.get(f'{proxy_url}/v1/models', headers={'Authorization': f'Bearer {proxy_key}'}, timeout=30)
models.raise_for_status()
model_ids = [item.get('id') for item in models.json().get('data', [])]
if model_alias not in model_ids:
    raise RuntimeError(f'Expected {model_alias} in /v1/models, got {model_ids}')
print('Health: OK')
print(f'Gateway models: {model_ids}')